# Dual-View Amulet AI — YOLOv11 Training Pipeline
**ก่อนกดรัน:** ตรวจสอบเมนูด้านบน **Runtime -> Change runtime type -> เลือก `T4 GPU`** แล้วกด Save

In [ ]:
!pip install -q ultralytics roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="RJu6VkIYQ5QiQYCfwnvA")
project = rf.workspace("al-by-ball").project("dual-view_amulet_ai")
version = project.version(1)
dataset = version.download("yolov11")

In [ ]:
import os
import glob
import yaml

TARGET_CLASSES = [
    "somdej_watrakang_front", "somdej_watrakang_back",
    "somdej_bangkhunprom_front", "somdej_bangkhunprom_back",
    "somdej_ketchaiyo_front", "somdej_ketchaiyo_back",
    "phra_nangphaya_front", "phra_nangphaya_back",
    "phra_phong_suphan_front", "phra_phong_suphan_back",
    "phra_rod_front", "phra_rod_back",
    "phra_sumkor_front", "phra_sumkor_back",
    "defect"
]

yaml_path = f"{dataset.location}/data.yaml"
with open(yaml_path, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

old_names = data_cfg["names"]
old_id_to_name = {int(k): v for k, v in old_names.items()} if isinstance(old_names, dict) else {i: n for i, n in enumerate(old_names)}
old_to_new_id = {old_id: TARGET_CLASSES.index(name) for old_id, name in old_id_to_name.items() if name in TARGET_CLASSES}

kept_images, removed_images = 0, 0

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(dataset.location, split, "images")
    lbl_dir = os.path.join(dataset.location, split, "labels")
    if not os.path.exists(img_dir):
        continue

    for img_path in glob.glob(os.path.join(img_dir, "*")):
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(lbl_dir, f"{base_name}.txt")

        new_lines = []
        if os.path.exists(lbl_path):
            with open(lbl_path, "r") as lf:
                for line in lf:
                    parts = line.strip().split()
                    if parts and int(parts[0]) in old_to_new_id:
                        parts[0] = str(old_to_new_id[int(parts[0])])
                        new_lines.append(" ".join(parts))

        if new_lines:
            with open(lbl_path, "w") as lf:
                lf.write("\n".join(new_lines) + "\n")
            kept_images += 1
        else:
            os.remove(img_path)
            if os.path.exists(lbl_path):
                os.remove(lbl_path)
            removed_images += 1

data_cfg["nc"] = len(TARGET_CLASSES)
data_cfg["names"] = TARGET_CLASSES
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_cfg, f, allow_unicode=True)

print(f"พร้อมเทรน! เหลือภาพที่มีป้ายเป้าหมาย {kept_images} ภาพ | ตัดภาพว่าง/นอกขอบเขตออก {removed_images} ภาพ")

In [ ]:
import os
from ultralytics import YOLO
from google.colab import files

model = YOLO("yolo11s.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16
)

best_weight_path = os.path.join(results.save_dir, "weights", "best.pt")
files.download(best_weight_path)